In [ ]:
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.neighbors import KNeighborsRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from bayes_opt import BayesianOptimization
import warnings
import random

warnings.filterwarnings('ignore')

random_state = 1
np.random.seed(random_state)
random.seed(random_state)

def fill_nan_with_median(data, name="数据"):
    """使用各列的中位数（数值列）或众数/默认值（非数值列）填充NaN"""
    if data.isnull().values.any():
        nan_count = data.isnull().sum().sum()
        print(f"警告：{name}包含 {nan_count} 个NaN值，将使用中位数/众数进行填充")
        
        num_cols = data.select_dtypes(include=[np.number]).columns
        for col in num_cols:
            if data[col].isnull().any():
                median_val = data[col].median()
                if pd.isna(median_val): median_val = 0.0
                data[col] = data[col].fillna(median_val)
                
        non_num_cols = data.select_dtypes(exclude=[np.number]).columns
        for col in non_num_cols:
            if data[col].isnull().any():
                mode_val = data[col].mode()
                fill_val = mode_val[0] if not mode_val.empty else 'missing'
                data[col] = data[col].fillna(fill_val)
                
        print(f"{name}：共填充了 {nan_count} 个NaN值")
    return data

def CV_group(data):
    """负载均衡的数据分组策略，将数据按ID均匀分配到5个组"""
    data = data.copy()
    data = fill_nan_with_median(data, "CV_group输入数据")
    
    data['ref_FO'] = data['ID'].map(str)
    ref_JQ = data['ID']
    tt_ion = pd.DataFrame(ref_JQ.value_counts(ascending=False))
    tt_ion_index = tt_ion.index

    one, two, three, four, five = pd.DataFrame(), pd.DataFrame(), pd.DataFrame(), pd.DataFrame(), pd.DataFrame()

    for i in range(len(tt_ion_index)):
        a = [len(one), len(two), len(three), len(four), len(five)]
        min_idx = a.index(min(a))
        current_data = data[data['ID'] == tt_ion_index[i]]
        
        if min_idx == 0: one = pd.concat([one, current_data])
        elif min_idx == 1: two = pd.concat([two, current_data])
        elif min_idx == 2: three = pd.concat([three, current_data])
        elif min_idx == 3: four = pd.concat([four, current_data])
        elif min_idx == 4: five = pd.concat([five, current_data])

    print(f"分组长度: one={len(one)}, two={len(two)}, three={len(three)}, four={len(four)}, five={len(five)}")
    return one, two, three, four, five

class EnsembleModel:
    """封装 XGBoost 和 KNN 的加权融合模型，提供统一的 predict 接口"""
    def __init__(self, xgb_model, knn_model, scaler_X, weight_xgb):
        self.xgb_model = xgb_model
        self.knn_model = knn_model
        self.scaler_X = scaler_X
        self.weight_xgb = weight_xgb
        self.is_fitted_ = True 
        
    def predict(self, X):
        if isinstance(X, pd.Series):
            X = X.to_frame().T
        elif isinstance(X, np.ndarray) and X.ndim == 1:
            X = X.reshape(1, -1)
            
        if isinstance(X, pd.DataFrame):
            X = X.values
            
        X_scaled = self.scaler_X.transform(X)
        pred_xgb = self.xgb_model.predict(X_scaled)
        pred_knn = self.knn_model.predict(X_scaled)
        
        return self.weight_xgb * pred_xgb + (1 - self.weight_xgb) * pred_knn

class ML_model():
    def __init__(self, train, test, one, two, three, four, five, features):
        self.features = features
        self.scaler_X = StandardScaler()
        
        for name, df in [("训练集", train), ("测试集", test), ("分组1", one), ("分组2", two), 
                         ("分组3", three), ("分组4", four), ("分组5", five)]:
            self._check_and_fill_data(df, name)
        
        self.X_train = pd.DataFrame(train)
        self.X_test = pd.DataFrame(test)
        
        if 'R' not in train.columns or 'R' not in test.columns:
            raise ValueError("数据必须包含'R'列作为目标变量")
        
        self.y_train = train['R']
        self.y_test = test['R']
        
        self.X_train_1 = pd.concat([two, three, four, five], join='outer')
        self.X_test_1 = one.copy()
        self.X_train_2 = pd.concat([one, three, four, five], join='outer')
        self.X_test_2 = two.copy()
        self.X_train_3 = pd.concat([one, two, four, five], join='outer')
        self.X_test_3 = three.copy()
        self.X_train_4 = pd.concat([one, two, three, five], join='outer')
        self.X_test_4 = four.copy()
        self.X_train_5 = pd.concat([one, two, three, four], join='outer')
        self.X_test_5 = five.copy()
        
        self._clean_cv_datasets()
        
        for i in range(1, 6):
            setattr(self, f'y_train_{i}', self._safe_get_target(getattr(self, f'X_train_{i}'), 'R', f'分组{i}训练集'))
            setattr(self, f'y_test_{i}', self._safe_get_target(getattr(self, f'X_test_{i}'), 'R', f'分组{i}测试集'))
            setattr(self, f'X_train_{i}', self._safe_select_features(getattr(self, f'X_train_{i}'), f"分组{i}训练集"))
            setattr(self, f'X_test_{i}', self._safe_select_features(getattr(self, f'X_test_{i}'), f"分组{i}测试集"))
            
        print("数据初始化完成，所有NaN问题已处理")
    
    def _check_and_fill_data(self, data, name):
        if data is None or len(data) == 0: return
        if data.isnull().sum().sum() > 0: fill_nan_with_median(data, name)
    
    def _clean_cv_datasets(self):
        for i in range(1, 6):
            for prefix, name in [('X_train', '训练集'), ('X_test', '测试集')]:
                df = getattr(self, f'{prefix}_{i}')
                if df is not None and df.isnull().sum().sum() > 0:
                    fill_nan_with_median(df, f"{name}{i}")
    
    def _safe_get_target(self, data, target_col, name):
        if target_col not in data.columns: raise ValueError(f"{name} 缺少目标列 '{target_col}'")
        y = data[target_col].copy()
        if y.isnull().any(): y = y.fillna(y.median())
        return y
    
    def _safe_select_features(self, data, name):
        X = data[self.features].copy()
        if X.isnull().values.any():
            for col in X.columns: X[col] = X[col].fillna(X[col].median())
        if np.isinf(X.values).any():
            X = X.replace([np.inf, -np.inf], np.nan)
            for col in X.columns: X[col] = X[col].fillna(X[col].median())
        return X
    
    def rem_name(self, org_data):
        if 'ref_ID' not in org_data.columns: return pd.DataFrame(index=org_data.index)
        return org_data.loc[:, ['ref_ID']].copy()
    
    def CV_train(self, params, X_train, y_train, X_test, y_test):
        """交叉验证训练方法 (XGBoost + KNN 融合版本)"""
        try:
            if X_train.empty or X_test.empty or len(y_train) == 0 or len(y_test) == 0:
                return float('inf')
            
            X_train_np = np.nan_to_num(X_train.values.astype(np.float32), nan=0.0, posinf=0.0, neginf=0.0)
            y_train_np = y_train.values.astype(np.float32)
            X_test_np = np.nan_to_num(X_test.values.astype(np.float32), nan=0.0, posinf=0.0, neginf=0.0)
            y_test_np = y_test.values.astype(np.float32)
            
            scaler_X = StandardScaler()
            X_train_scaled = scaler_X.fit_transform(X_train_np)
            X_test_scaled = scaler_X.transform(X_test_np)
            
            xgb_model = xgb.XGBRegressor(
                n_estimators=int(round(params['xgb_n_estimators'])),
                max_depth=int(round(params['xgb_max_depth'])),
                learning_rate=float(params['xgb_learning_rate']),
                subsample=float(params['xgb_subsample']),
                colsample_bytree=0.8,
                reg_alpha=0.1, reg_lambda=1.0,
                random_state=random_state, n_jobs=-1, verbosity=0
            )
            xgb_model.fit(X_train_scaled, y_train_np)
            
            knn_weights = 'distance' if params['knn_weights'] > 0.5 else 'uniform'
            knn_model = KNeighborsRegressor(
                n_neighbors=int(round(params['knn_n_neighbors'])),
                weights=knn_weights,
                p=2,
                n_jobs=-1
            )
            knn_model.fit(X_train_scaled, y_train_np)
            
            weight_xgb = float(params['weight_xgb'])
            pred_xgb = xgb_model.predict(X_test_scaled)
            pred_knn = knn_model.predict(X_test_scaled)
            y_pred = weight_xgb * pred_xgb + (1 - weight_xgb) * pred_knn
            
            y_pred = np.nan_to_num(y_pred, nan=0.0, posinf=0.0, neginf=0.0)
            y_test_clean = np.nan_to_num(y_test_np, nan=0.0, posinf=0.0, neginf=0.0)
            
            rmse = np.sqrt(mean_squared_error(y_test_clean, y_pred))
            return rmse
            
        except Exception as e:
            print(f"集成模型训练过程中出错: {str(e)}")
            return float('inf')
    
    def data_prep_for_model(self, params):
        """执行五折交叉验证"""
        fold_results = [
            self.CV_train(params, getattr(self, f'X_train_{i}'), getattr(self, f'y_train_{i}'), 
                          getattr(self, f'X_test_{i}'), getattr(self, f'y_test_{i}'))
            for i in range(1, 6)
        ]
        
        valid_rmses = [rmse for rmse in fold_results if not np.isinf(rmse) and not np.isnan(rmse)]
        if len(valid_rmses) == 0: return -1
        
        rmse_mean = np.mean(valid_rmses)
        print(f"五折RMSE: {[round(r, 4) for r in fold_results]} | 平均RMSE: {rmse_mean:.4f}")
        return -1 * rmse_mean 
    
    def ensemble_evaluate(self, xgb_n_estimators, xgb_max_depth, xgb_learning_rate, xgb_subsample,
                          knn_n_neighbors, knn_weights, weight_xgb):
        """集成模型超参数评估函数"""
        params = {
            'xgb_n_estimators': xgb_n_estimators, 'xgb_max_depth': xgb_max_depth,
            'xgb_learning_rate': xgb_learning_rate, 'xgb_subsample': xgb_subsample,
            'knn_n_neighbors': knn_n_neighbors, 'knn_weights': knn_weights,
            'weight_xgb': weight_xgb
        }
        return self.data_prep_for_model(params)
    
    def param(self):
        """执行贝叶斯优化超参数搜索"""
        pbounds = {
            'xgb_n_estimators': (50, 500),
            'xgb_max_depth': (3, 10),
            'xgb_learning_rate': (0.01, 0.3),
            'xgb_subsample': (0.6, 1.0),
            'knn_n_neighbors': (3, 30),
            'knn_weights': (0, 1),
            'weight_xgb': (0.1, 0.9)
        }
        
        bo = BayesianOptimization(
            f=self.ensemble_evaluate,
            pbounds=pbounds,
            verbose=2,
            random_state=random_state
        )
        
        print('=' * 20)
        print('Starting Bayesian Optimization for XGBoost + KNN Ensemble...')
        print('=' * 20)
        
        try:
            bo.maximize(init_points=10, n_iter=500) 
        except Exception as e:
            print(f"贝叶斯优化出错: {str(e)}")
            return {'xgb_n_estimators': 100, 'xgb_max_depth': 6, 'xgb_learning_rate': 0.1, 
                    'xgb_subsample': 0.8, 'knn_n_neighbors': 5, 'knn_weights': 0.0, 'weight_xgb': 0.7}
        
        best_params = bo.max['params']
        print(f"最佳超参数: {best_params}")
        return best_params
    
    def final_model(self, X_train, y_train):
        """构建最终的集成模型"""
        try:
            params = self.param()
            
            X_train_np = np.nan_to_num(X_train[self.features].values.astype(np.float32), nan=0.0, posinf=0.0, neginf=0.0)
            y_train_np = y_train.values.astype(np.float32)
            
            self.scaler_X.fit(X_train_np)
            X_train_scaled = self.scaler_X.transform(X_train_np)
            
            print("\n训练最终的 XGBoost 模型...")
            xgb_model = xgb.XGBRegressor(
                n_estimators=int(round(params['xgb_n_estimators'])),
                max_depth=int(round(params['xgb_max_depth'])),
                learning_rate=float(params['xgb_learning_rate']),
                subsample=float(params['xgb_subsample']),
                colsample_bytree=0.8, reg_alpha=0.1, reg_lambda=1.0,
                random_state=random_state, n_jobs=-1, verbosity=0
            )
            xgb_model.fit(X_train_scaled, y_train_np)
            
            print("训练最终的 KNN 模型...")
            knn_weights = 'distance' if params['knn_weights'] > 0.5 else 'uniform'
            knn_model = KNeighborsRegressor(
                n_neighbors=int(round(params['knn_n_neighbors'])),
                weights=knn_weights, p=2, n_jobs=-1
            )
            knn_model.fit(X_train_scaled, y_train_np)
            
            weight_xgb = float(params['weight_xgb'])
            print(f"模型融合完成！XGBoost权重: {weight_xgb:.2f}, KNN权重: {1-weight_xgb:.2f}")
            
            return EnsembleModel(xgb_model, knn_model, self.scaler_X, weight_xgb)
            
        except Exception as e:
            print(f"训练最终模型时出错: {str(e)}，使用默认参数...")
            xgb_m = xgb.XGBRegressor(n_estimators=100, max_depth=6, learning_rate=0.1, random_state=random_state)
            knn_m = KNeighborsRegressor(n_neighbors=5)
            xgb_m.fit(self.scaler_X.transform(X_train_np), y_train_np)
            knn_m.fit(self.scaler_X.transform(X_train_np), y_train_np)
            return EnsembleModel(xgb_m, knn_m, self.scaler_X, 0.7)
    
    def result(self, model):
        """模型性能评估 (直接调用 EnsembleModel 的 predict 方法)"""
        try:
            X_train_name = self.rem_name(self.X_train)
            X_test_name = self.rem_name(self.X_test)
            
            y_train, y_test = self.y_train, self.y_test
            
            X_train_np = np.nan_to_num(self.X_train[self.features].values.astype(np.float32), nan=0.0, posinf=0.0, neginf=0.0)
            X_test_np = np.nan_to_num(self.X_test[self.features].values.astype(np.float32), nan=0.0, posinf=0.0, neginf=0.0)
            
            y_train_pred = model.predict(X_train_np)
            y_pred = model.predict(X_test_np)
            
            y_train_pred = np.nan_to_num(y_train_pred, nan=0.0, posinf=0.0, neginf=0.0)
            y_pred = np.nan_to_num(y_pred, nan=0.0, posinf=0.0, neginf=0.0)
            y_train_clean = np.nan_to_num(y_train.values, nan=0.0, posinf=0.0, neginf=0.0)
            y_test_clean = np.nan_to_num(y_test.values, nan=0.0, posinf=0.0, neginf=0.0)
            
            train_r2, train_rmse, train_mae = r2_score(y_train_clean, y_train_pred), np.sqrt(mean_squared_error(y_train_clean, y_train_pred)), mean_absolute_error(y_train_clean, y_train_pred)
            test_r2, test_rmse, test_mae = r2_score(y_test_clean, y_pred), np.sqrt(mean_squared_error(y_test_clean, y_pred)), mean_absolute_error(y_test_clean, y_pred)
            
            print("\n" + "=" * 60)
            print("XGBoost + KNN 集成模型性能结果")
            print("=" * 60)
            print(f"【训练集】 R²: {train_r2:.4f} | RMSE: {train_rmse:.4f} | MAE: {train_mae:.4f}")
            print(f"【验证集】 R²: {test_r2:.4f} | RMSE: {test_rmse:.4f} | MAE: {test_mae:.4f}")
            print("=" * 60)
            
            y_pred_df = pd.DataFrame(y_pred, columns=['pred'], index=self.X_test.index)
            y_test_df = pd.DataFrame(y_test_clean, columns=['Js'], index=self.X_test.index)
            y_train_pred_df = pd.DataFrame(y_train_pred, columns=['pred'], index=self.X_train.index)
            
            total = pd.concat([X_train_name, self.X_train[self.features], pd.Series(y_train_clean, name='Js', index=self.X_train.index)], axis=1)
            total_train = pd.concat([total, y_train_pred_df], axis=1)
            
            total_test = pd.concat([X_test_name, self.X_test[self.features], y_test_df, y_pred_df], axis=1)
            
            total_train['error'] = total_train['Js'] - total_train['pred']
            total_test['error'] = total_test['Js'] - total_test['pred']
            
            return self, total_train, total_test, total_train[['Js', 'pred', 'error']], total_test[['Js', 'pred', 'error']]
            
        except Exception as e:
            print(f"评估过程中出错: {str(e)}")
            return self, pd.DataFrame(), pd.DataFrame(), pd.DataFrame(), pd.DataFrame()


try:
    df = pd.read_excel('FO xgboost.xlsx', sheet_name='data')
    print("成功加载Excel文件")
except Exception as e:
    print(f"加载Excel文件失败: {str(e)}，使用示例数据...")
    np.random.seed(42)
    n_samples = 200
    df = pd.DataFrame({
        'ID': np.random.randint(1, 10, n_samples),
        'MPDC': np.random.normal(0, 1, n_samples), 'TMCC': np.random.normal(0, 1, n_samples),
        'MPDIPT': np.random.normal(0, 1, n_samples), 'TMCIPT': np.random.normal(0, 1, n_samples),
        'MT': np.random.normal(0, 1, n_samples), 'PTE': np.random.normal(0, 1, n_samples),
        'PTI': np.random.normal(0, 1, n_samples), 'OPS': np.random.normal(0, 1, n_samples),
        'WP': np.random.normal(0, 1, n_samples), 'MO': np.random.normal(0, 1, n_samples),
        'DST': np.random.normal(0, 1, n_samples), 'DSC': np.random.normal(0, 1, n_samples),
        'ST': np.random.normal(0, 1, n_samples), 'CV': np.random.normal(0, 1, n_samples),
        'Jw': np.random.normal(50, 20, n_samples), 'Js': np.random.normal(0, 1, n_samples),
        'R': np.random.normal(50, 20, n_samples), 'ref_ID': np.random.randint(1, 5, n_samples)
    })

expected_columns = ['ID', 'MPDC', 'TMCC', 'MPDIPT', 'TMCIPT', 'MT', 'PTE', 'PTI', 'OPS',
                   'WCA', 'WP', 'MO', 'DST', 'DSC', 'ST', 'Jw', 'Js', 'R', 'ref_ID']

if len(df.columns) == len(expected_columns): df.columns = expected_columns
df.index = range(len(df))

features = ['MPDC', 'TMCC', 'MPDIPT', 'TMCIPT', 'MT', 'PTE', 'PTI', 'OPS',
            'WCA', 'WP', 'MO', 'DST', 'DSC', 'ST',]
features = [col for col in features if col in df.columns]

if 'ref_ID' in df.columns:
    ref_id = pd.to_numeric(df['ref_ID'], errors='coerce')
    train, test = df[ref_id != 1].copy(), df[ref_id == 1].copy()
    if len(test) == 0:
        from sklearn.model_selection import train_test_split
        train, test = train_test_split(df, test_size=0.2, random_state=42)
else:
    from sklearn.model_selection import train_test_split
    train, test = train_test_split(df, test_size=0.2, random_state=42)

print(f"训练集大小: {len(train)}, 测试集大小: {len(test)}")
df = fill_nan_with_median(df, "全局数据集df")

try:
    one, two, three, four, five = CV_group(train.copy())
except Exception as e:
    from sklearn.model_selection import KFold
    kf = KFold(n_splits=5, shuffle=True, random_state=42)
    splits = list(kf.split(train))
    one, two, three, four, five = [train.iloc[splits[i][1]] for i in range(5)]

ml_model = ML_model(train, test, one, two, three, four, five, features)
final_model = ml_model.final_model(train, train['R'])
result, total_train, total_test, train_error, test_error = ml_model.result(final_model)

try:
    import plotly.express as px
    import plotly.graph_objects as go
    
    if not total_train.empty and not total_test.empty:
        total_train['mark'] = 'train'
        total_test['mark'] = 'test'
        total_data = pd.concat([total_train, total_test])
    
        X_title, Y_title = 'Real R (%)', 'Pred R (%)'
        fig = px.scatter(total_data, x='Js', y='pred', color='mark', width=700, height=600,
                         labels={'Js': X_title, 'pred': Y_title},
                         color_discrete_map={'train': '1f77b4', 'test': 'ff7f0e'})
    
        fig.update_layout(
            xaxis=dict(title_text=X_title, title_font={"size": 25}, tickfont=dict(size=18), title_standoff=20, ticks="inside"),
            yaxis=dict(title_text=Y_title, title_font={"size": 25}, tickfont=dict(size=18), title_standoff=20, ticks="inside"),
            plot_bgcolor='rgb(256,256,256)', showlegend=True
        )
        fig.update_xaxes(showline=True, linewidth=2, linecolor='black', mirror=True)
        fig.update_yaxes(showline=True, linewidth=2, linecolor='black', mirror=True)
    
        min_val = min(total_data['Js'].min(), total_data['pred'].min())
        max_val = max(total_data['Js'].max(), total_data['pred'].max())
        fig.add_trace(go.Scatter(x=[min_val, max_val], y=[min_val, max_val], 
                                 line=dict(color='red', width=2, dash='dash'), name='Perfect Prediction'))
    
        fig.update_traces(marker=dict(size=8, opacity=0.7, line=dict(width=0.5)), selector=dict(mode='markers'))
        fig.show()
except Exception as e:
    print(f"生成可视化图表出错: {str(e)}")

print("\n程序执行完成！")

print("\n程序执行完成！")

In [ ]:
try:
    import shap
    import matplotlib.pyplot as plt
    import matplotlib
    matplotlib.use('Agg')
    import os

    shap_dir = 'shap_output'
    os.makedirs(shap_dir, exist_ok=True)
    print(f"\n{'=' * 60}")
    print(f"SHAP 可解释性分析开始")
    print(f"{'=' * 60}")

    xgb_model_final = final_model.xgb_model
    scaler_final = final_model.scaler_X
    weight_xgb_final = final_model.weight_xgb
    weight_knn_final = 1 - weight_xgb_final

    X_train_shap = pd.DataFrame(
        scaler_final.transform(
            np.nan_to_num(train[features].values.astype(np.float32), nan=0.0, posinf=0.0, neginf=0.0)
        ),
        columns=features, index=train.index
    )
    X_test_shap = pd.DataFrame(
        scaler_final.transform(
            np.nan_to_num(test[features].values.astype(np.float32), nan=0.0, posinf=0.0, neginf=0.0)
        ),
        columns=features, index=test.index
    )

    print(f"SHAP 分析数据准备完成：训练集 {X_train_shap.shape}，测试集 {X_test_shap.shape}")
    print(f"模型融合权重 → XGBoost: {weight_xgb_final:.2f}, KNN: {weight_knn_final:.2f}")

    print("\n>>> 正在计算 XGBoost 的 SHAP 值（TreeExplainer）...")
    
    import json
    try:
        booster = xgb_model_final.get_booster()
        config = json.loads(booster.save_config())
        base_score_str = config['learner']['learner_model_param'].get('base_score', '0.5')
        if isinstance(base_score_str, str) and base_score_str.startswith('['):
            import ast
            base_score_val = ast.literal_eval(base_score_str)
            if isinstance(base_score_val, list) and len(base_score_val) > 0:
                config['learner']['learner_model_param']['base_score'] = str(base_score_val[0])
                booster.load_config(json.dumps(config))
    except Exception: pass

    explainer_xgb = shap.TreeExplainer(xgb_model_final)

    shap_values_train = explainer_xgb.shap_values(X_train_shap)
    shap_values_test = explainer_xgb.shap_values(X_test_shap)
    
    if hasattr(shap_values_train, 'values'):
        shap_values_train_arr = shap_values_train.values
        shap_values_test_arr = shap_values_test.values
    else:
        shap_values_train_arr = shap_values_train
        shap_values_test_arr = shap_values_test

    def safe_float(val):
        if isinstance(val, np.ndarray): return float(val.flatten()[0])
        return float(val)

    base_val_train = safe_float(explainer_xgb.expected_value)
    base_val_test = base_val_train

    print("SHAP 值计算完成！")
    print(f"  训练集 SHAP values shape: {shap_values_train_arr.shape}")
    print(f"  测试集 SHAP values shape: {shap_values_test_arr.shape}")
    print(f"  基准值 (E[f(x)]): {base_val_train:.4f}")

    print("\n>>> 生成图1：SHAP Summary Plot（蜂群图）...")
    fig, axes = plt.subplots(1, 2, figsize=(22, 8))

    plt.sca(axes[0])
    shap.summary_plot(shap_values_train_arr, X_train_shap, show=False, max_display=len(features))
    plt.title('SHAP Summary Plot - Training Set', fontsize=16, fontweight='bold', pad=15)

    plt.sca(axes[1])
    shap.summary_plot(shap_values_test_arr, X_test_shap, show=False, max_display=len(features))
    plt.title('SHAP Summary Plot - Test Set', fontsize=16, fontweight='bold', pad=15)
    
    plt.tight_layout()
    plt.savefig(f'{shap_dir}/01_shap_summary_beeswarm.png', dpi=300, bbox_inches='tight')
    plt.savefig(f'{shap_dir}/01_shap_summary_beeswarm.svg', format='svg', bbox_inches='tight')
    plt.close('all')
    print("  ✅ 已保存: 01_shap_summary_beeswarm.png / .svg")

    print(">>> 生成图2：SHAP 特征重要性条形图...")
    fig, axes = plt.subplots(1, 2, figsize=(20, 8))

    plt.sca(axes[0])
    shap.summary_plot(shap_values_train_arr, X_train_shap, plot_type="bar", show=False, max_display=len(features))
    plt.title('Feature Importance (Mean |SHAP|) - Training Set', fontsize=14, fontweight='bold')

    plt.sca(axes[1])
    shap.summary_plot(shap_values_test_arr, X_test_shap, plot_type="bar", show=False, max_display=len(features))
    plt.title('Feature Importance (Mean |SHAP|) - Test Set', fontsize=14, fontweight='bold')
    
    plt.tight_layout()
    plt.savefig(f'{shap_dir}/02_shap_bar_importance.png', dpi=300, bbox_inches='tight')
    plt.savefig(f'{shap_dir}/02_shap_bar_importance.svg', format='svg', bbox_inches='tight')
    plt.close('all')
    print("  ✅ 已保存: 02_shap_bar_importance.png / .svg")

    print(">>> 生成图3：SHAP Violin Plot...")
    fig, ax = plt.subplots(figsize=(12, 8))
    shap.summary_plot(shap_values_test_arr, X_test_shap, plot_type="violin", show=False, max_display=len(features))
    plt.title('SHAP Violin Plot - Test Set', fontsize=16, fontweight='bold', pad=15)
    plt.tight_layout()
    plt.savefig(f'{shap_dir}/03_shap_violin_plot.png', dpi=300, bbox_inches='tight')
    plt.savefig(f'{shap_dir}/03_shap_violin_plot.svg', format='svg', bbox_inches='tight')
    plt.close('all')
    print("  ✅ 已保存: 03_shap_violin_plot.png / .svg")

    print(">>> 生成图4：SHAP Dependence Plot（Top 4 特征）...")
    mean_abs_shap = np.abs(shap_values_test_arr).mean(axis=0)
    top_indices = np.argsort(mean_abs_shap)[::-1][:4]
    top_features = [features[i] for i in top_indices]
    print(f"  Top 4 特征: {top_features}")

    fig, axes = plt.subplots(2, 2, figsize=(18, 14))
    for idx, feat_name in enumerate(top_features):
        ax = axes[idx // 2][idx % 2]
        plt.sca(ax)
        shap.dependence_plot(feat_name, shap_values_test_arr, X_test_shap, show=False, ax=ax)
        ax.set_title(f'Dependence Plot: {feat_name}', fontsize=14, fontweight='bold')
    plt.suptitle('SHAP Dependence Plots - Top 4 Features (Test Set)', fontsize=18, fontweight='bold', y=1.02)
    plt.tight_layout()
    plt.savefig(f'{shap_dir}/04_shap_dependence_top4.png', dpi=300, bbox_inches='tight')
    plt.savefig(f'{shap_dir}/04_shap_dependence_top4.svg', format='svg', bbox_inches='tight')
    plt.close('all')
    print("  ✅ 已保存: 04_shap_dependence_top4.png / .svg")

    print(">>> 生成图5：SHAP Waterfall Plot（测试集前 3 个样本）...")
    n_waterfall = min(3, len(X_test_shap))
    for i in range(n_waterfall):
        fig, ax = plt.subplots(figsize=(12, 8))
        explanation_single = shap.Explanation(
            values=shap_values_test_arr[i],
            base_values=base_val_test,
            data=X_test_shap.iloc[i].values,
            feature_names=features
        )
        shap.plots.waterfall(explanation_single, show=False, max_display=len(features))
        plt.title(f'Waterfall Plot - Test Sample {i} (Index: {X_test_shap.index[i]})', 
                  fontsize=14, fontweight='bold', pad=15)
        plt.tight_layout()
        plt.savefig(f'{shap_dir}/05_shap_waterfall_sample_{i}.png', dpi=300, bbox_inches='tight')
        plt.savefig(f'{shap_dir}/05_shap_waterfall_sample_{i}.svg', format='svg', bbox_inches='tight')
        plt.close('all')
        print(f"  ✅ 已保存: 05_shap_waterfall_sample_{i}.png / .svg")

    print(">>> 生成图6：SHAP Force Plot（力图）...")
    try:
        shap.initjs()
        
        shap.save_html(
            f'{shap_dir}/06_shap_force_plot_all_test.html',
            base_val_test,
            shap_values_test_arr,
            X_test_shap
        )
        print("  ✅ 已保存: 06_shap_force_plot_all_test.html")

        for i in range(min(3, len(X_test_shap))):
            shap.save_html(
                f'{shap_dir}/06_shap_force_plot_sample_{i}.html',
                base_val_test,
                shap_values_test_arr[i],
                X_test_shap.iloc[i]
            )
        print("  ✅ 已保存: 06_shap_force_plot_sample_0/1/2.html")
    except Exception as e:
        print(f"  ⚠️ Force Plot 生成失败: {e}")

    print(">>> 生成图7：SHAP Heatmap（热力图）...")
    sample_importance = np.abs(shap_values_test_arr).sum(axis=1)
    top_sample_idx = np.argsort(sample_importance)[::-1][:50]
    
    shap_exp_test = shap.Explanation(
        values=shap_values_test_arr[top_sample_idx],
        base_values=np.full(len(top_sample_idx), base_val_test),
        data=X_test_shap.iloc[top_sample_idx].values,
        feature_names=features
    )

    fig, ax = plt.subplots(figsize=(14, 10))
    shap.plots.heatmap(shap_exp_test, show=False, max_display=len(features))
    plt.title('SHAP Heatmap - Top 50 Most Influential Test Samples', fontsize=16, fontweight='bold', pad=15)
    plt.tight_layout()
    plt.savefig(f'{shap_dir}/07_shap_heatmap.png', dpi=300, bbox_inches='tight')
    plt.savefig(f'{shap_dir}/07_shap_heatmap.svg', format='svg', bbox_inches='tight')
    plt.close('all')
    print("  ✅ 已保存: 07_shap_heatmap.png / .svg")

    print(">>> 生成图8：SHAP Interaction Values（交互效应）...")
    try:
        n_interaction = min(500, len(X_test_shap))
        X_interaction = X_test_shap.iloc[:n_interaction]
        
        interaction_values = explainer_xgb.shap_interaction_values(X_interaction)
        mean_interaction = np.abs(interaction_values).mean(axis=0)

        fig, ax = plt.subplots(figsize=(12, 10))
        import seaborn as sns
        interaction_df = pd.DataFrame(mean_interaction, index=features, columns=features)
        sns.heatmap(
            interaction_df, annot=True, fmt='.4f', cmap='YlOrRd',
            xticklabels=features, yticklabels=features, ax=ax, 
            linewidths=0.5, square=True, cbar_kws={'label': 'Mean |Interaction Value|'}
        )
        ax.set_title('SHAP Interaction Values - Feature Interaction Matrix', fontsize=16, fontweight='bold', pad=15)
        plt.xticks(rotation=45, ha='right', fontsize=11)
        plt.yticks(rotation=0, fontsize=11)
        plt.tight_layout()
        plt.savefig(f'{shap_dir}/08_shap_interaction_matrix.png', dpi=300, bbox_inches='tight')
        plt.savefig(f'{shap_dir}/08_shap_interaction_matrix.svg', format='svg', bbox_inches='tight')
        plt.close('all')
        print(f"  ✅ 已保存: 08_shap_interaction_matrix.png / .svg")
    except Exception as e:
        print(f"  ⚠️ Interaction Values 计算失败: {e}")

    print(">>> 生成图9：特征正负 SHAP 贡献对比图...")
    try:
        mean_shap = shap_values_test_arr.mean(axis=0)
        sorted_idx = np.argsort(np.abs(mean_shap))[::-1]

        fig, ax = plt.subplots(figsize=(12, 8))
        colors = ['d62728' if v > 0 else '1f77b4' for v in mean_shap[sorted_idx]]
        ax.barh(range(len(features)), mean_shap[sorted_idx], color=colors, edgecolor='black', linewidth=0.5)
        ax.set_yticks(range(len(features)))
        ax.set_yticklabels([features[i] for i in sorted_idx], fontsize=12)
        ax.set_xlabel('Mean SHAP Value (signed)', fontsize=14)
        ax.set_title('Average SHAP Contribution per Feature (Test Set)\nRed = Positive Push | Blue = Negative Push', fontsize=16, fontweight='bold', pad=15)
        ax.axvline(x=0, color='black', linewidth=1.2, linestyle='-')
        ax.grid(axis='x', alpha=0.3)
        plt.gca().invert_yaxis()
        plt.tight_layout()
        plt.savefig(f'{shap_dir}/09_shap_signed_contribution.png', dpi=300, bbox_inches='tight')
        plt.savefig(f'{shap_dir}/09_shap_signed_contribution.svg', format='svg', bbox_inches='tight')
        plt.close('all')
        print("  ✅ 已保存: 09_shap_signed_contribution.png / .svg")
    except Exception as e:
        print(f"  ⚠️ 正负贡献图生成失败: {e}")

    print(">>> 生成图10：SHAP Decision Plot（决策路径）...")
    try:
        fig, ax = plt.subplots(figsize=(12, 10))
        n_decision = min(20, len(X_test_shap))
        shap.multioutput_decision_plot(
            base_val_test,
            shap_values_test_arr[:n_decision],
            feature_names=features,
            highlight=list(range(min(3, n_decision))),
            legend_label=[f'Sample {i}' for i in range(n_decision)],
            show=False
        )
        plt.title('SHAP Decision Plot - Test Samples Decision Path', fontsize=16, fontweight='bold', pad=15)
        plt.tight_layout()
        plt.savefig(f'{shap_dir}/10_shap_decision_plot.png', dpi=300, bbox_inches='tight')
        plt.savefig(f'{shap_dir}/10_shap_decision_plot.svg', format='svg', bbox_inches='tight')
        plt.close('all')
        print("  ✅ 已保存: 10_shap_decision_plot.png / .svg")
    except Exception as e:
        print(f"  ⚠️ Decision Plot 生成失败: {e}")

    print(">>> 导出 SHAP 数值结果到 CSV...")
    try:
        shap_df_test = pd.DataFrame(shap_values_test_arr, columns=[f'SHAP_{f}' for f in features], index=X_test_shap.index)
        shap_df_test['base_value'] = base_val_test
        shap_df_test['predicted_value'] = shap_df_test[[c for c in shap_df_test.columns if c.startswith('SHAP_')]].sum(axis=1) + shap_df_test['base_value']
        shap_df_test['actual_R'] = test['R'].values[:len(shap_df_test)]
        shap_df_test.to_csv(f'{shap_dir}/shap_values_test.csv', index=True, encoding='utf-8-sig')

        shap_df_train = pd.DataFrame(shap_values_train_arr, columns=[f'SHAP_{f}' for f in features], index=X_train_shap.index)
        shap_df_train['base_value'] = base_val_train
        shap_df_train['predicted_value'] = shap_df_train[[c for c in shap_df_train.columns if c.startswith('SHAP_')]].sum(axis=1) + shap_df_train['base_value']
        shap_df_train['actual_R'] = train['R'].values[:len(shap_df_train)]
        shap_df_train.to_csv(f'{shap_dir}/shap_values_train.csv', index=True, encoding='utf-8-sig')

        importance_df = pd.DataFrame({
            'Feature': features,
            'Mean_SHAP_abs_test': np.abs(shap_values_test_arr).mean(axis=0),
            'Mean_SHAP_signed_test': shap_values_test_arr.mean(axis=0),
            'Std_SHAP_test': shap_values_test_arr.std(axis=0),
        })
        importance_df = importance_df.sort_values('Mean_SHAP_abs_test', ascending=False).reset_index(drop=True)
        importance_df['Importance_Rank_test'] = range(1, len(importance_df) + 1)
        importance_df.to_csv(f'{shap_dir}/shap_feature_importance_summary.csv', index=False, encoding='utf-8-sig')

        print("  ✅ 已保存: shap_values_test.csv, shap_values_train.csv, shap_feature_importance_summary.csv")
    except Exception as e:
        print(f"  ⚠️ CSV 导出失败: {e}")

    print(f"\n{'=' * 60}")
    print("📊 SHAP 特征重要性排名（测试集，按 Mean |SHAP| 排序）")
    print(f"{'=' * 60}")
    print(f"{'Rank':<6}{'Feature':<15}{'Mean|SHAP|':<14}{'Mean SHAP':<14}{'Std SHAP':<14}")
    print(f"{'-' * 63}")
    for i in range(len(importance_df)):
        row = importance_df.iloc[i]
        print(f"{int(row['Importance_Rank_test']):<6}{row['Feature']:<15}"
              f"{float(row['Mean_SHAP_abs_test']):<14.6f}{float(row['Mean_SHAP_signed_test']):<14.6f}"
              f"{float(row['Std_SHAP_test']):<14.6f}")
    print(f"{'=' * 60}")

    print(f"\n{'=' * 60}")
    print(f"🎉 SHAP 分析完成！所有结果已保存到 '{shap_dir}/' 目录")
    print(f"{'=' * 60}")
    print("生成的文件清单：")
    for f in sorted(os.listdir(shap_dir)):
        fsize = os.path.getsize(os.path.join(shap_dir, f))
        print(f"  📄 {f:<50} ({fsize / 1024:.1f} KB)")
    print(f"{'=' * 60}")

except ImportError:
    print("\n⚠️ 未安装 SHAP 库！请运行: pip install shap")
except Exception as e:
    print(f"\n⚠️ SHAP 分析过程中出错: {str(e)}")
    import traceback
    traceback.print_exc()

In [ ]:
try:
    from IPython.display import display, Image, SVG, HTML, IFrame
    import os

    print(f"\n{'=' * 60}")
    print("🖼️ 开始在 Notebook 中内联渲染 SHAP 分析图表")
    print(f"{'=' * 60}\n")

    display_list = [
        ("01_shap_summary_beeswarm.svg", "图1：SHAP Summary Plot（蜂群图 - 特征重要性、方向与分布）"),
        ("02_shap_bar_importance.svg", "图2：SHAP 特征全局重要性条形图（Mean |SHAP|）"),
        ("03_shap_violin_plot.svg", "图3：SHAP Violin Plot（小提琴图 - 特征值分布密度）"),
        ("04_shap_dependence_top4.svg", "图4：SHAP Dependence Plot（Top 4 特征非线性依赖与交互效应）"),
        ("05_shap_waterfall_sample_0.svg", "图5：SHAP Waterfall Plot（瀑布图 - 单样本决策解释）"),
        ("07_shap_heatmap.svg", "图7：SHAP Heatmap（热力图 - 样本×特征 SHAP 矩阵）"),
        ("08_shap_interaction_matrix.svg", "图8：SHAP Interaction Values（特征交互效应矩阵）"),
        ("09_shap_signed_contribution.svg", "图9：特征正负 SHAP 贡献对比图（总体推拉方向）"),
        ("10_shap_decision_plot.svg", "图10：SHAP Decision Plot（多样本决策累积路径图）"),
    ]

    for filename, title in display_list:
        filepath = os.path.join(shap_dir, filename)
        if os.path.exists(filepath):
            display(HTML(f"<h3 style='color:2c3e50; border-bottom:2px solid 3498db; padding-bottom:5px;'>{title}</h3>"))
            if filename.endswith('.svg'):
                display(SVG(filename=filepath))
            else:
                display(Image(filename=filepath, retina=True))
            display(HTML("<br>"))
        else:
            display(HTML(f"<p style='color:gray;'>⚠️ 未找到文件: {filename}</p>"))

    force_plot_file = os.path.join(shap_dir, '06_shap_force_plot_all_test.html')
    if os.path.exists(force_plot_file):
        display(HTML("<h3 style='color:2c3e50; border-bottom:2px solid e74c3c; padding-bottom:5px;'>图6：SHAP Force Plot（全局交互式力图）</h3>"))
        display(HTML("<p style='color:555;'>💡 <b>交互提示</b>：鼠标悬停查看具体特征贡献，可使用顶部下拉菜单切换样本，或拖动横轴缩放。</p>"))
        
        with open(force_plot_file, 'r', encoding='utf-8') as f:
            html_content = f.read()
        
        try:
            display(IFrame(src=force_plot_file, width="100%", height=450))
        except Exception:
            display(HTML(html_content))
            
        display(HTML("<br>"))

    print(f"{'=' * 60}")
    print("🎉 所有 SHAP 图表已在上方渲染完毕！")
    print(f"{'=' * 60}")

except ImportError:
    print("\n⚠️ 未安装 IPython！如果在 Jupyter 中运行，请执行: pip install ipython")
except Exception as e:
    print(f"\n⚠️ 图表内联显示过程中出错: {str(e)}")

In [ ]:
import joblib
import json
import os
import pandas as pd
import numpy as np

os.makedirs('saved_models', exist_ok=True)

joblib.dump(final_model, 'saved_models/model_R.pkl')

features = ['MPDC', 'TMCC', 'MPDIPT', 'TMCIPT', 'MT', 'PTE', 'PTI', 'OPS', 
            'WCA', 'WP', 'MO', 'DST', 'DSC', 'ST', ]

bounds = {}
for feat in features:
    bounds[feat] = [float(df[feat].min()), float(df[feat].max())]

with open('saved_models/feature_bounds.json', 'w') as f:
    json.dump(bounds, f, indent=4)
    
print("模型和特征边界保存成功！")